In [0]:
%sql
SELECT * FROM content_job.silver.account_details

In [0]:
df = spark.read.table("content_job.gold.account_master_view")
df.display()

In [0]:
%sql
SELECT * FROM content_job.temp.df_sha256_account_details

In [0]:
%sql
SELECT userId, SUM(friendsCount) FROM content_job.temp.df_sha256_account_details 
GROUP BY userId
ORDER BY 2 DESC
LIMIT 10

In [0]:
%sql
--DROP TABLE content_job.silver.account_details

In [0]:
%sql
SELECT * FROM content_job.temp.df_sha256_account_details

In [0]:
%sql
--SELECT * FROM content_job.gold.post_popularity_score
SELECT verificationConfidence, COUNT(*) 
FROM content_job.gold.account_master_view 
GROUP BY verificationConfidence

In [0]:
%sql
SELECT DISTINCT accountAgeCategory FROM content_job.temp.df_sha256_account_details

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table("content_job.bronze.account_details")
df.withColumn("accountMetadata.accountAge.accountAgeCategory", F.when(F.col("accountMetadata.accountAge.createdYear").between(2016,2019), "Long-term")).display()

In [0]:
%sql
SELECT DISTINCT LEFT(account_creation_year_month, 4) FROM content_job.temp.df_sha256_account_details ORDER BY 1 DESC

In [0]:
%sql
SELECT is_group, COUNT(*) FROM content_job.gold.account_master_view
GROUP BY is_group
--SELECT potentialBot, COUNT(*) FROM content.target.gold_account_master_view GROUP BY potentialBot

In [0]:
%sql
SELECT accountAgeCategory, COUNT(*) FROM content_job.gold.account_master_view
GROUP BY accountAgeCategory

In [0]:
df = spark.read.table("content_job.temp.df_sha256_account_details")
print(df.columns)

In [0]:
%sql
SELECT * FROM content_job.temp.df_sha256_account_details

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType

json_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("friendsCount", IntegerType(), True),
    StructField("listedCount", IntegerType(), True),
    StructField("location", StringType(), True),
    StructField("rawDescription", StringType(), True),
    # Zagnieżdżona struktura accountMetadata
    StructField("accountMetadata", StructType([
        StructField("accountAge", StructType([
            StructField("createdYear", StringType(), True),
            StructField("createdMonth", StringType(), True),
            StructField("accountAgeCategory", StringType(), True)
        ])),
        StructField("verificationStatus", StructType([
            StructField("isVerified", BooleanType(), True),
            StructField("verificationConfidence", DoubleType(), True)
        ]))
    ])),
    # Zagnieżdżona struktura analyticsFlags
    StructField("analyticsFlags", StructType([
        StructField("potentialBot", IntegerType(), True),
        StructField("potentialInfluencer", IntegerType(), True)
    ])),
    # Zagnieżdżona struktura profileAnalysis
    StructField("profileAnalysis", StructType([
        StructField("profileCompletenessScore", DoubleType(), True)
    ])),
    # Zagnieżdżona struktura networkFeatures
    StructField("networkFeatures", StructType([
        StructField("networkType", StringType(), True)
    ]))
])


In [0]:
def bronze_account_details():
        return (
            spark.read
           
            .format("json")
            #.option('format','delta')

            .schema(json_schema)
            #.option("cloudFiles.inferColumnTypes","true")
            .option("multiline","true")
            .load("/Volumes/content/landing/json_files_data//*.json")
            .select(
                "*",
                F.current_timestamp().alias("ingest_time")
))


stg = bronze_account_details()
stg.display()

In [0]:
stg.groupBy("analyticsFlags.potentialBot").count().show()

In [0]:
%sql
SELECT * FROM content_job.silver.account_details

In [0]:
%sql
SELECT potentialBot, COUNT(*) FROM content_job.silver.account_details
GROUP BY potentialBot


In [0]:
%sql
GRANT ALL PRIVILEGES ON CATALOG content_job TO `dbadmin@patrykwalat7gmail.onmicrosoft.com`;

In [0]:
def bronze_():
    df = spark.read.format("jdbc") \
        .option("url", "jdbc:sqlserver://sql.database.windows.net;databaseName=database") \
        .option("username", "x") \
        .option("password", "x") \
        .option("dbtable", "cdc.SalesLT_Customer_CT") \
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
        .load()
    return (df)

In [0]:
%sql
SELECT src_null.*, NULL AS mergeKey FROM content_job.temp.df_sha256_hashtags src_null
       LEFT JOIN content_job.silver.hashtags tgt ON src_null.hashtag_id = tgt.hashtag_id-- AND tgt.is_current = True
       WHERE tgt.hashtag_id IS NULL

In [0]:
%sql
--- ROWS THAT HAD CHANGED
SELECT src.*, src.hashtag_id AS mergeKey FROM content_job.temp.df_sha256_hashtags src
JOIN content_job.silver.hashtags tgt ON src.hashtag_id = tgt.hashtag_id 
WHERE tgt.is_current = true AND tgt.sha_key <> src.sha_key

In [0]:
%sql

SELECT src_null.*, NULL AS mergeKey, tgt.* FROM content_job.temp.df_sha256_hashtags src_null
       LEFT JOIN content_job.silver.hashtags tgt ON src_null.hashtag_id = tgt.hashtag_id AND tgt.is_current = true
       WHERE tgt.hashtag_id IS NULL


  UNION ALL

       --- ROWS THAT HAD CHANGED
       SELECT src.*, src.hashtag_id AS mergeKey, tgt.* FROM content_job.temp.df_sha256_hashtags src
       JOIN content_job.silver.hashtags tgt ON src.hashtag_id = tgt.hashtag_id 
       WHERE tgt.is_current = true AND tgt.sha_key <> src.sha_key


In [0]:
%sql
SELECT * FROM content_job.silver.hashtags WHERE hashtag_id = 400

In [0]:
%sql

 --- INSERTING COMPLETELY NEW ROWS
       SELECT src_null.*, NULL AS mergeKey, 'INSERT' AS action FROM content_job.temp.df_sha256_hashtags src_null
       LEFT JOIN content_job.silver.hashtags tgt ON src_null.hashtag_id = tgt.hashtag_id AND tgt.is_current = true
       WHERE tgt.hashtag_id IS NULL OR tgt.sha_key <> src_null.sha_key
      
       UNION ALL

       --- ROWS THAT HAD CHANGED
       SELECT src.*, src.hashtag_id AS mergeKey, 'UPDATE' AS action FROM content_job.temp.df_sha256_hashtags src
       JOIN content_job.silver.hashtags tgt ON src.hashtag_id = tgt.hashtag_id 
       WHERE tgt.is_current = true AND tgt.sha_key <> src.sha_key

       UNION ALL
       -- DELETED ROWS
       SELECT tgt.hashtag_id, tgt.tag_text, tgt.first_use_time, tgt.valid_from, tgt.sha_key, tgt.hashtag_id AS mergeKey, 'DELETE' AS action
       FROM content_job.silver.hashtags tgt
       LEFT JOIN content_job.temp.df_sha256_hashtags src ON tgt.hashtag_id = src.hashtag_id 
       WHERE src.hashtag_id IS NULL AND tgt.is_current = true


In [0]:
%sql
USE CATALOG content_job;
USE SCHEMA bronze;

In [0]:
%sql

--SELECT * FROM content_job.bronze.hashtags WHERE hashtag_id = 400;
--SELECT * FROM content_job.temp.df_sha256_hashtags WHERE hashtag_id = 400;
SELECT * FROM content_job.silver.hashtags WHERE hashtag_id = 400 ORDER BY 1 ASC


In [0]:
%sql
--DROP TABLE content_job.temp.df_sha256_account_details

In [0]:
dbutils.fs.rm("/content/landing/checkpoints/bronze_acc_details_json", True)

In [0]:

dbutils.fs.rm("/content/landing/checkpoints/df_sha_account_details", True)

In [0]:
%sql
SELECT src_null.*, NULL AS mergeKey FROM content_job.temp.df_sha256_hashtags src_null
       LEFT JOIN content_job.silver.hashtags tgt ON src_null.hashtag_id = tgt.hashtag_id AND tgt.is_current = true
       WHERE tgt.hashtag_id IS NULL

In [0]:
%sql
SELECT * FROM content_job.temp.df_sha256_follow_relationship

In [0]:
%sql
DROP TABLE content_job.bronze.follow_relationship;
DROP TABLE content_job.temp.df_sha256_follow_relationship;
DROP TABLE content_job.silver.follow_relationship;

In [0]:
%sql
SELECT src_null.*, NULL AS mergeKey 
    FROM content_job.temp.df_sha256_account_details AS src_null
    LEFT JOIN content_job.silver.account_details tgt ON src_null.userId = tgt.userId 
    WHERE tgt.userId IS NULL

In [0]:
%sql
WITH CTE AS (

  SELECT src_null.*, NULL AS mergeKey FROM content_job.temp.df_sha256_account_user src_null
UNION ALL


SELECT src.*, src.account_id AS mergeKey FROM content_job.temp.df_sha256_account_user src
JOIN content_job.silver.account_user tgt ON tgt.account_id = src.account_id AND tgt.is_current = True
WHERE tgt.sha_key <> src.sha_key

)
SELECT * FROM CTE WHERE account_id = 1

In [0]:
%sql
SELECT src.*, src.account_id AS mergeKey FROM content_job.temp.df_sha256_account_user src
JOIN content_job.silver.account_user tgt ON tgt.account_id = src.account_id AND tgt.is_current = True
WHERE tgt.sha_key <> src.sha_key

In [0]:
%sql
DROP TABLE content_job.bronze.account_details;
DROP TABLE content_job.temp.df_sha256_account_details;

In [0]:
%sql
SELECT COUNT(*) FROM content_job.silver.account_user
--WHERE account_id = 1

In [0]:
df = spark.table("gold.gold_account_master_view")
df.display()

In [0]:
%sql
CREATE SCHEMA content_job.temp

In [0]:
%sql
DROP TABLE content_job.temp.df_sha256_advertisements;
DROP TABLE content_job.bronze.advertisements;

In [0]:
%sql
SHOW VOLUMES

In [0]:
def bronze_account_user():
    return (
        spark.read
        .format("jdbc") 
        .option("url", dbutils.secrets.get(scope="sm-secret-scope", key = "social-media-project-db-jdbc")) 
        .option("username", dbutils.secrets.get(scope="sm-secret-scope", key = "social-media-project-dblog")) 
        .option("password", dbutils.secrets.get(scope="sm-secret-scope", key = "social-media-project-secret")) 
        .option("dbtable", dbutils.secrets.get(scope="sm-secret-scope", key = "social-media-project-db-tab-acc-users")) 
        .load()
        .display()
    )

In [0]:
%sql
DELETE FROM content_job.temp.df_sha256_account_details

In [0]:
%sql
DESCRIBE DETAIL content_job.bronze.bronze_account_user

In [0]:
display(dbutils.fs.ls("/mnt/content_job/bronze/bronze_account_user"))

In [0]:
%sql
SELECT * FROM content_job.silver.advertisements
WHERE advertisement_id = 500

In [0]:
%sql
--TRUNCATE TABLE content_job.silver.account_user_scd_type2

In [0]:
%sql
--UPDATE content_job.bronze.bronze_account_user SET is_group = false WHERE account_id = 1379

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType

json_schema = StructType([
    StructField("userId", StringType(), True),
    StructField("friendsCount", IntegerType(), True),
    StructField("listedCount", IntegerType(), True),
    StructField("location", StringType(), True),
    StructField("rawDescription", StringType(), True),
    # Zagnieżdżona struktura accountMetadata
    StructField("accountMetadata", StructType([
        StructField("accountAge", StructType([
            StructField("createdYear", StringType(), True),
            StructField("createdMonth", StringType(), True),
            StructField("accountAgeCategory", StringType(), True)
        ])),
        StructField("verificationStatus", StructType([
            StructField("isVerified", BooleanType(), True),
            StructField("verificationConfidence", DoubleType(), True)
        ]))
    ])),
    # Zagnieżdżona struktura analyticsFlags
    StructField("analyticsFlags", StructType([
        StructField("potentialBot", BooleanType(), True),
        StructField("potentialInfluencer", BooleanType(), True)
    ])),
    # Zagnieżdżona struktura profileAnalysis
    StructField("profileAnalysis", StructType([
        StructField("profileCompletenessScore", DoubleType(), True)
    ])),
    # Zagnieżdżona struktura networkFeatures
    StructField("networkFeatures", StructType([
        StructField("networkType", StringType(), True)
    ]))
])


# Here I'm using ba
def bronze_account_details():
        return (
            spark.read#Stream
           # .format("cloudFiles")
            .format("json")
            #.option('format','delta')
            #.option("cloudFiles.format", "json")
            .schema(json_schema)
            #.option("cloudFiles.inferColumnTypes","true")
            .option("multiline","true")
            .load("/Volumes/content/landing/json_files_data/Test/")
            .select(
                "*",
                F.current_timestamp().alias("ingest_time")
))



stg = bronze_account_details()#.filter(F.col("userId").isNotNull())
stg.display()

In [0]:
stg = stg.select(
                "userId",
                F.date_format(F.make_date(F.col("accountMetadata.accountAge.createdYear").cast("int"), F.col("accountMetadata.accountAge.createdMonth").cast("int"), F.lit(1)),'yyyy-MM').alias("account_creation_year_month"),
                "accountMetadata.accountAge.accountAgeCategory",
                "accountMetadata.verificationStatus.isVerified",
                "accountMetadata.verificationStatus.verificationConfidence",
                "analyticsFlags.potentialBot",
                "analyticsFlags.potentialInfluencer",
                "friendsCount",
                "listedCount",
                "location",
                "rawDescription",
                "profileAnalysis.profileCompletenessScore",
                "networkFeatures.networkType",
                "ingest_time"
            )
            #.withColumn("accountAgeCategory", F.regexp_replace("accountAgeCategory", "_", ' ')) \
            #.withColumn("networkType", F.regexp_replace("networkType", "_", ' '))


stg = (stg
        .withColumn("accountAgeCategory", F.regexp_replace("accountAgeCategory", "_", ' '))
        .withColumn("networkType", F.regexp_replace("networkType", "_", ' '))
)

stg.display()

In [0]:
%sql
SELECT COUNT(*) FROM content_job.bronze.bronze_account_user

In [0]:
%sql
DROP TABLE content_job.temp.dfsha256_follow_relationship

In [0]:
%sql
--DROP TABLE content_job.bronze.bronze_account_details;
--DROP TABLE content_job.bronze.bronze_account_user;
--DROP TABLE content_job.bronze.time;
--DROP TABLE content_job.temp.df_sha256_time;
--DROP TABLE content_job.temp.df_sha256_account_user;
--DROP TABLE content_job.temp.df_sha256_account_details;
--DROP TABLE content_job.gold.gold_account_master_view;
DROP TABLE content_job.silver.account_user_scd_type2;
DROP TABLE content_job.silver.silver_account_details;
--DROP TABLE content_job.silver.time

In [0]:
%sql
--DROP TABLE content_job.silver.advertisers

In [0]:
%sql
SELECT * FROM content_job.silver.advertisers

In [0]:
%sql
USE SCHEMA silver

In [0]:
%sql
SELECT acc.account_id, 
           acc.account_name, 
           acc.is_group, 
           det.accountAgeCategory, 
           det.profileCompletenessScore, 
           det.friendsCount, 
           det.isVerified,
           det.verificationConfidence, 
           det.potentialInfluencer, 
           det.potentialBot 
FROM content_job.silver.account_user_scd_type2 acc 
LEFT JOIN content_job.silver.silver_account_details det ON acc.account_id = det.userId
WHERE acc.is_current = True
AND acc.account_id = 1
--AND det.is_current = True

### POST_HASHTAG

In [0]:
%sql
--DROP TABLE content_job.silver.post_hashtag

In [0]:
%sql
  SELECT src_null.*, tgt.*, NULL AS mergeKey_1, NULL AS mergeKey_2,'INSERT' AS action
    FROM content_job.temp.df_sha256_post_hashtag src_null 
    LEFT JOIN content_job.silver.post_hashtag tgt 
    ON src_null.post_id = tgt.post_id AND src_null.hashtag_id = tgt.hashtag_id --AND tgt.is_current = true
    WHERE tgt.post_id IS NULL AND tgt.hashtag_id IS NULL

In [0]:
%sql
SELECT * FROM content_job.silver.post_hashtag ORDER BY 1 ASC

In [0]:
%sql
SELECT src_null.*, tgt.*, NULL AS mergeKey_1, NULL AS mergeKey_2,'INSERT' AS action
    FROM content_job.temp.df_sha256_post_hashtag src_null 
    LEFT JOIN content_job.silver.post_hashtag tgt 
    ON src_null.post_id = tgt.post_id AND src_null.hashtag_id = tgt.hashtag_id --AND tgt.is_current = true
    WHERE tgt.post_id IS NULL AND tgt.hashtag_id IS NULL

In [0]:
%sql
SELECT * FROM content_job.silver.post_hashtag ORDER BY 1 ASC

In [0]:
%sql

 -- COMPLETELY NEW ROWS
    SELECT src_null.*, tgt.*, NULL AS mergeKey_1, NULL AS mergeKey_2,'INSERT' AS action
    FROM content_job.temp.df_sha256_post_hashtag src_null 
    LEFT JOIN content_job.silver.post_hashtag tgt 
    ON src_null.post_id = tgt.post_id AND src_null.hashtag_id = tgt.hashtag_id
    WHERE tgt.post_id IS NULL AND tgt.hashtag_id IS NULL 

    UNION ALL 
      
    -- CHANGED ROWS
    SELECT src.* , tgt.*, src.post_id AS mergeKey_1, src.hashtag_id AS mergeKey_2, 'UPDATE' AS action
    FROM content_job.temp.df_sha256_post_hashtag src 
    JOIN content_job.silver.post_hashtag tgt ON src.post_id = tgt.post_id AND src.hashtag_id = tgt.hashtag_id AND tgt.is_current = true
    WHERE src.sha_key <> tgt.sha_key


In [0]:
%sql
/*
    -- COMPLETELY NEW ROWS
    SELECT src_null.*, NULL AS mergeKey_1, NULL AS mergeKey_2,'INSERT' AS action
    FROM content_job.temp.df_sha256_post_hashtag src_null 
    LEFT JOIN content_job.silver.post_hashtag tgt 
    ON src_null.post_id = tgt.post_id AND src_null.hashtag_id = tgt.hashtag_id
    WHERE tgt.post_id IS NULL AND tgt.hashtag_id IS NULL
    
    UNION ALL 

    -- CHANGED ROWS
    SELECT src.* , src.post_id AS mergeKey_1, src.hashtag_id AS mergeKey_2, 'UPDATE' AS action
    FROM content_job.temp.df_sha256_post_hashtag src 
    JOIN content_job.silver.post_hashtag tgt ON src.post_id = tgt.post_id AND src.hashtag_id = tgt.hashtag_id AND tgt.is_current
    WHERE src.sha_key <> tgt.sha_key

    -- DELETE ROWS
    UNION ALL
*/
    SELECT tgt.post_id, tgt.hashtag_id, tgt.valid_from, tgt.sha_key, tgt.post_id AS mergeKey_1, tgt.hashtag_id AS mergeKey_2, 'DELETE' AS action
    FROM content_job.silver.post_hashtag tgt 
    LEFT JOIN content_job.temp.df_sha256_post_hashtag src 
    ON src.post_id = tgt.post_id AND src.hashtag_id = tgt.hashtag_id AND tgt.is_current = true
    WHERE src.post_id IS NULL AND src.hashtag_id IS NULL

In [0]:
%sql
SELECT * FROM content_job.silver.post_hashtag ORDER BY 1 ASC

In [0]:
%sql
SELECT * FROM content_job.bronze.post_hashtag ORDER BY 1 ASC

### content_job.silver.comments

In [0]:
%sql
DROP TABLE content_job.silver.comments;

In [0]:
%sql
SELECT * FROM content_job.silver.comments

In [0]:
%sql
-- INSERT NEW ROWS
SELECT src_null.*, tgt.*, NULL AS mergeKey, 'INSERT' AS action FROM content_job.temp.df_sha256_comments src_null
  LEFT JOIN content_job.silver.comments tgt ON src_null.comment_id = tgt.comment_id 
  WHERE tgt.comment_id IS NULL 

In [0]:
%sql
SELECT * FROM content_job.silver.comments ORDER BY 1 ASC

TEN KOD DOBRZE DZIAŁA DLA SAMYCH NOWYCH REKORDOW

SELECT src_null.*, NULL AS mergeKey, 'INSERT' AS action FROM content_job.temp.df_sha256_comments src_null
        LEFT JOIN content_job.silver.comments tgt ON src_null.comment_id = tgt.comment_id 
        WHERE tgt.comment_id IS NULL 



In [0]:
%sql

--- INSERT NEW ROWS
        SELECT src_null.*, NULL AS mergeKey, 'INSERT' AS action FROM content_job.temp.df_sha256_comments src_null
        LEFT JOIN content_job.silver.comments tgt ON src_null.comment_id = tgt.comment_id 
        WHERE tgt.comment_id IS NULL OR src_null.sha_key <> tgt.sha_key

        UNION ALL

        -- ROWS THAT CHANGED IN THE PAST
        SELECT src.*, src.comment_id AS mergeKey, 'UPDATE' AS action FROM content_job.temp.df_sha256_comments src
        JOIN content_job.silver.comments tgt ON src.comment_id = tgt.comment_id
        WHERE tgt.is_current = true AND src.sha_key <> tgt.sha_key


In [0]:
%sql
SELECT * FROM content_job.silver.comments ORDER BY 1 ASC

In [0]:
%sql
SELECT * FROM content_job.temp.df_sha256_comments

In [0]:
%sql

 --- INSERT NEW ROWS 
        SELECT src_null.*, NULL AS mergeKey, tgt.*,'INSERT' AS action FROM content_job.temp.df_sha256_comments src_null
        LEFT JOIN content_job.silver.comments tgt ON src_null.comment_id = tgt.comment_id --AND tgt.is_current = true
        WHERE tgt.comment_id IS NULL OR src_null.sha_key <> tgt.sha_key
/*
        UNION ALL

        -- ROWS THAT CHANGED IN THE PAST
        SELECT src.*, src.comment_id AS mergeKey, 'UPDATE' AS action FROM content_job.temp.df_sha256_comments src
        JOIN content_job.silver.comments tgt ON src.comment_id = tgt.comment_id
        WHERE tgt.is_current = true AND src.sha_key <> tgt.sha_key
*/

In [0]:
%sql
--- DELETE ROWS
SELECT tgt.comment_id, tgt.author_account_id, tgt.post_id, tgt.created_at_time, tgt.comment_text, tgt.status,
tgt.is_image, tgt.valid_from, tgt.sha_key, tgt.comment_id AS mergeKEy, 'DELETE' AS action FROM content_job.silver.comments tgt
LEFT JOIN content_job.bronze.comments src ON tgt.comment_id = src.comment_id 
WHERE tgt.is_current = true AND src.comment_id IS NULL

### Reactions

In [0]:
%sql
--DROP TABLE content_job.silver.reactions

In [0]:
%sql
SELECT * FROM content_job.temp.df_sha256_reactions

In [0]:
%sql
SELECT * FROM content_job.silver.reactions ORDER BY 1 ASC

In [0]:
%sql
SELECT acc.account_id, 
        acc.account_name, 
        acc.is_group, 
        det.accountAgeCategory, 
        det.profileCompletenessScore, 
        det.friendsCount, 
        det.isVerified,
        det.verificationConfidence, 
        det.potentialInfluencer, 
        det.potentialBot 
FROM content_job.silver.account_user acc 
LEFT JOIN content_job.silver.account_details det ON acc.account_id = det.userId
WHERE acc.is_current = True
AND det.is_current = True